In [1]:
import polars as pl

import torch
from datasets import Dataset

from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments

import logging
from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

# Configure logging levels to hide model-loading report
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

disable_progress_bar()

In [2]:
# Q1. Label Encoding
# Convert the answer column in train.csv into numeric labels using the following mapping:
# A = 0
# B = 1
# C = 2
# D = 3
# E = 4

# What is the encoded numeric label for the row at index 150?

data = pl.read_csv("../data/train.csv")

label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
data = data.with_columns(
    pl.col("answer").replace(label_mapping).cast(pl.Int64).alias("label")
)
q1_answer = data["label"][150]

print(f"Q1 Answer: {q1_answer}")

Q1 Answer: 2


In [3]:
# Q2. Prompt-Option Formatting
# For row index 0, create the Option B input using exactly this format:
# str(prompt) + " [SEP] " + str(option_B)

# What is the exact character length of this formatted input string?

row_0 = data.row(0, named=True)

formatted_string_B = str(row_0["prompt"]) + " [SEP] " + str(row_0["B"])
q2_answer = len(formatted_string_B)

print(f"Q2 Answer: {q2_answer}")

Q2 Answer: 407


In [4]:
# Q3. Single-Row MCQ Tokenization
# Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
# padding = "max_length"
# truncation = True
# max_length = 128
# return_tensors = "pt"

# After reshaping for a multiple-choice model, the final input_ids tensor has shape:
# [1, 5, 128]

# What is the value of the second dimension?

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

choices = [
    str(row_0["prompt"]) + " [SEP] " + str(row_0["A"]),
    str(row_0["prompt"]) + " [SEP] " + str(row_0["B"]),
    str(row_0["prompt"]) + " [SEP] " + str(row_0["C"]),
    str(row_0["prompt"]) + " [SEP] " + str(row_0["D"]),
    str(row_0["prompt"]) + " [SEP] " + str(row_0["E"]),
]

tokens_q3 = tokenizer(
    choices, 
    padding="max_length", 
    truncation=True, 
    max_length=128, 
    return_tensors="pt"
)

input_ids = tokens_q3["input_ids"].unsqueeze(0)
attention_mask = tokens_q3["attention_mask"].unsqueeze(0)

q3_answer = input_ids.shape[1]
print(f"Q3 Answer: {q3_answer}")

Q3 Answer: 5


In [5]:
# Q4. Batch MCQ Tokenization
# Tokenize the first 16 rows of train.csv as multiple-choice examples.
# Each row has 5 choices.
# Each choice is tokenized to length 128.

# The final input_ids tensor has shape:
# [16, 5, 128]

# How many total token positions are in this tensor?

q4_answer = 16 * 5 * 128 # 16 rows * 5 options * 128 length

print(f"Q4 Answer (Total token positions): {q4_answer}")

Q4 Answer (Total token positions): 10240


In [6]:
# Q5. Multiple-Choice Logits
# Load bert-base-uncased using AutoModelForMultipleChoice.
# Tokenize row index 0 as 5 choices and pass it through the model.

# The output logits tensor has shape:
# [1, 5]

# How many logits are produced for one question?

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

labels = torch.tensor([row_0["label"]])
outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

logits = outputs.logits
q5_answer = logits.shape[1] 

print(f"Q5 Answer: {q5_answer}")

Q5 Answer: 5


In [7]:
# Q6. Supervised Loss Tensor
# For row index 0, pass the tokenized 5-choice input into 
# AutoModelForMultipleChoice along with the correct encoded label.

# The model returns a scalar loss tensor.

# How many dimensions does this loss tensor have?

loss = outputs.loss
q6_answer = loss.dim()

print(f"Q6 Answer: {q6_answer}")

Q6 Answer: 0


In [8]:
# Q7. LoRA Trainable Parameters
# Apply LoRA to the bert-base-uncased multiple-choice model using:
# r = 8
# lora_alpha = 16
# target_modules = ["query", "value"]
# lora_dropout = 0.1
# bias = "none"
# task_type = TaskType.SEQ_CLS

# Count trainable parameters using:
# sum(p.numel() for p in model.parameters() if p.requires_grad)

# How many parameters are trainable?

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

peft_model = get_peft_model(model, peft_config)
q7_answer = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

print(f"Q7 Answer: {q7_answer}")

Q7 Answer: 295681


In [9]:
# Q8. Hugging Face Dataset Preparation
# Create a Hugging Face Dataset from the first 100 rows of train.csv.

# For each row, create:
# input_ids with shape [5, 128]
# attention_mask with shape [5, 128]
# labels as the encoded answer label

# For the first dataset item, input_ids has shape:
# [5, 128]

# How many tokenized choices are stored in input_ids?

data_100 = data.head(100)

def preprocess_fn(examples: Dataset, max_len: int = 128):
    prompts = examples["prompt"]
    batch_size = len(prompts)
    
    flattened_choices = []
    for i in range(batch_size):
        for opt in ["A", "B", "C", "D", "E"]:
            flattened_choices.append(str(prompts[i]) + " [SEP] " + str(examples[opt][i]))
            
    tokenized = tokenizer(
        flattened_choices, 
        padding="max_length", 
        truncation=True, 
        max_length=max_len
    )
    
    return {
        "input_ids": [tokenized["input_ids"][i:i+5] for i in range(0, len(flattened_choices), 5)],
        "attention_mask": [tokenized["attention_mask"][i:i+5] for i in range(0, len(flattened_choices), 5)],
        "labels": examples["label"]
    }

hf_dataset_100 = Dataset.from_dict(data_100.to_dict(as_series=False))
encoded_dataset_100 = hf_dataset_100.map(preprocess_fn, batched=True, remove_columns=hf_dataset_100.column_names)

q8_answer = len(encoded_dataset_100[0]["input_ids"]) 

print(f"Q8 Answer: {q8_answer}")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Q8 Answer: 5


In [10]:
# Q9. Tiny LoRA Fine-Tuning
# Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

# Use the following settings:
# max_length = 64
# per_device_train_batch_size = 4
# gradient_accumulation_steps = 1
# max_steps = 4

# What is the final global_step reported by the Trainer?

data_32 = data.head(32)
hf_dataset_32 = Dataset.from_dict(data_32.to_dict(as_series=False))

encoded_dataset_32 = hf_dataset_32.map(
    lambda x: preprocess_fn(x, max_len=64), 
    batched=True, 
    remove_columns=hf_dataset_32.column_names
)

encoded_dataset_32.set_format("torch")

training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=encoded_dataset_32
)

trainer.train()

q9_answer = trainer.state.global_step
print(f"Q9 Answer: {q9_answer}")

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

{'train_runtime': '12.24', 'train_samples_per_second': '1.307', 'train_steps_per_second': '0.327', 'train_loss': '1.625', 'epoch': '0.5'}
Q9 Answer: 4


In [11]:
# Q10. Probability Assigned to Option E After Fine-Tuning
# Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

# What is the probability assigned to Option E?

# Round your answer to 4 decimal places.

peft_model.eval()

row_0_tensors = encoded_dataset_32[0]
input_ids_64 = row_0_tensors["input_ids"].unsqueeze(0) 
attention_mask_64 = row_0_tensors["attention_mask"].unsqueeze(0)

with torch.no_grad():
    outputs_ft = peft_model(input_ids=input_ids_64, attention_mask=attention_mask_64)

probabilities = torch.nn.functional.softmax(outputs_ft.logits, dim=-1)

q10_answer = probabilities[0, 4].item()

print(f"Q10 Answer: {q10_answer:.4f}")

Q10 Answer: 0.2004
